# Filter Response Visualization

Stage 3 filter sanity plots: low-pass magnitude response, envelope response, and cutoff modulation. These plots mirror the constants and simple envelope math in `Source/DSP/FilterConfig.h` and `Source/DSP/Filter.cpp`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample_rate = 48_000
cutoff_min_hz = 200.0
cutoff_max_hz = 20_000.0
cutoff_default_hz = 12_000.0
attack_ms = 10.0
release_ms = 150.0
depth_octaves = 0.5

def detector_coefficient(time_ms):
    seconds = max(0.001, time_ms) * 0.001
    return np.exp(-1.0 / (sample_rate * seconds))

def envelope_follower(signal):
    attack = detector_coefficient(attack_ms)
    release = detector_coefficient(release_ms)
    envelope = np.zeros_like(signal)
    state = 0.0
    for index, sample in enumerate(np.abs(signal)):
        coefficient = attack if sample > state else release
        state = coefficient * state + (1.0 - coefficient) * sample
        envelope[index] = state
    return envelope

def modulated_cutoff(base_cutoff_hz, envelope, depth_octaves):
    cutoff = base_cutoff_hz * np.power(2.0, -depth_octaves * np.clip(envelope, 0.0, 1.0))
    return np.clip(cutoff, cutoff_min_hz, sample_rate * 0.49)

def one_pole_lowpass_response(cutoff_hz, freqs):
    # Approximate visualization only; production uses JUCE TPT SVF.
    coefficient = 1.0 - np.exp(-2.0 * np.pi * cutoff_hz / sample_rate)
    z = np.exp(-1j * 2.0 * np.pi * freqs / sample_rate)
    response = coefficient / (1.0 - (1.0 - coefficient) * z)
    return 20.0 * np.log10(np.maximum(np.abs(response), 1e-12))

In [ ]:
freqs = np.geomspace(20.0, 20_000.0, 1024)
plt.figure(figsize=(10, 4))
for cutoff in [500.0, 1_000.0, 3_000.0, 12_000.0, 20_000.0]:
    plt.semilogx(freqs, one_pole_lowpass_response(cutoff, freqs), label=f'{cutoff:g} Hz')
plt.title('Low-pass response sketch (production uses JUCE TPT SVF)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.ylim(-36, 3)
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.tight_layout()

In [ ]:
duration = 2.0
time = np.arange(int(sample_rate * duration)) / sample_rate
source = np.zeros_like(time)
source[(time >= 0.2) & (time < 0.8)] = 1.0
source[(time >= 1.1) & (time < 1.35)] = 0.6
env = envelope_follower(source)
cutoff = modulated_cutoff(cutoff_default_hz, env, depth_octaves)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(time, source, label='input magnitude')
axes[0].plot(time, env, label='envelope')
axes[0].set_title('Envelope follower response')
axes[0].set_ylabel('Level')
axes[0].legend()
axes[1].plot(time, cutoff)
axes[1].set_title('Cutoff ducking from 12 kHz baseline')
axes[1].set_ylabel('Cutoff (Hz)')
axes[1].set_xlabel('Time (s)')
plt.tight_layout()

In [ ]:
envelopes = np.linspace(0.0, 1.0, 200)
plt.figure(figsize=(8, 4))
for depth in [0.0, 0.5, 1.0, 2.0]:
    plt.plot(envelopes, modulated_cutoff(cutoff_default_hz, envelopes, depth), label=f'{depth:g} oct')
plt.title('Envelope depth mapping')
plt.xlabel('Normalized envelope')
plt.ylabel('Cutoff (Hz)')
plt.legend(title='Depth')
plt.tight_layout()